# 🧾 Tax Document Intelligence System
## AI-Powered Receipt & Invoice Data Extraction

---

## 🎯 Project Overview

**Tax Document Intelligence** is an end-to-end NLP system that automatically extracts financial data from invoices, receipts, and tax documents using:

- 📸 **Computer Vision (EasyOCR)** — Extract text from any receipt or invoice image
- 🤖 **DistilBERT Classifier** — Automatically classify document type
- 🔍 **Smart Entity Extraction** — Pull out vendor, amount, date, invoice number
- 🗄️ **SQLite Database** — Store and manage all extracted records
- 🌐 **Gradio Interface** — Beautiful interactive demo

---

## 🏆 Key Features

1. 📄 **Receipt Scanner** — Upload any receipt or invoice image
2. 🏷️ **Document Classifier** — Auto-detect: Invoice / Receipt / Tax Form
3. 💰 **Financial Extractor** — Vendor, total, date, invoice number, line items
4. 🗄️ **Database Storage** — Every extraction saved to SQLite
5. 📊 **Records Viewer** — Browse all extracted documents
6. 🎨 **Professional UI** — Clean Gradio interface

---

**Built for:** NLP & MLOps Final Year Project — Bootcamp Cohort 14  
**Stack:** EasyOCR · DistilBERT · FastAPI · SQLite · Gradio  
**Deployment:** HuggingFace Spaces → https://huggingface.co/spaces/Attiqfr/tax-document-intelligence

---
**Let's extract some financial data! 🚀**

---
# Part 1: Environment Setup
Install all required packages. Takes ~2 minutes on first run.

In [ ]:
# ============================================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================================

print("📦 Installing Tax Document Intelligence Dependencies...")
print("="*70)
print("⏱️  This takes ~2 minutes on first run\n")

# OCR Engine
print("1/4 Installing EasyOCR (fast, accurate)...")
!pip install easyocr -q

# Computer Vision
print("2/4 Installing OpenCV & image processing...")
!pip install opencv-python-headless pillow -q

# NLP / Transformers
print("3/4 Installing Transformers & PyTorch...")
!pip install transformers torch -q

# Database & Interface
print("4/4 Installing SQLite tools & Gradio...")
!pip install gradio sqlalchemy -q

# PDF support
!pip install pymupdf -q

print("\n" + "="*70)
print("✅ All packages installed successfully!")
print("="*70 + "\n")

---
# Part 2: Import Libraries

In [ ]:
# ============================================================================
# CELL 2: IMPORT LIBRARIES
# ============================================================================

print("📚 Importing libraries...\n")

import os
import re
import json
import time
import sqlite3
import warnings
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List

warnings.filterwarnings('ignore')

# Computer Vision & OCR
import easyocr
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

# Deep Learning
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Database
import sqlalchemy
from sqlalchemy import create_engine, Column, Integer, String, Float, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker

# Interface
import gradio as gr

print("✅ All libraries imported!")
print(f"   PyTorch version  : {torch.__version__}")
print(f"   Transformers     : OK")
print(f"   EasyOCR          : OK")
print(f"   Gradio           : {gr.__version__}")
print("\n📍 Ready to initialize components.")

---
# Part 3: Database Setup
SQLite database to store all extracted document records.

In [ ]:
# ============================================================================
# CELL 3: DATABASE SETUP
# ============================================================================

print("🗄️  Setting up SQLite Database...\n")

Base = declarative_base()

class Document(Base):
    __tablename__ = "documents"
    id              = Column(Integer, primary_key=True, autoincrement=True)
    filename        = Column(String(255))
    document_type   = Column(String(100))
    vendor          = Column(String(255))
    invoice_number  = Column(String(100))
    date            = Column(String(50))
    total_amount    = Column(String(50))
    subtotal        = Column(String(50))
    tax_amount      = Column(String(50))
    raw_text        = Column(Text)
    items           = Column(Text)   # JSON string
    confidence      = Column(Float)
    created_at      = Column(DateTime, default=datetime.utcnow)

# Create DB
engine  = create_engine("sqlite:///tax_documents.db", echo=False)
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

def save_to_db(data: dict) -> int:
    session = Session()
    try:
        doc = Document(
            filename       = data.get("filename", "uploaded_image"),
            document_type  = data.get("document_type", "unknown"),
            vendor         = data.get("vendor", ""),
            invoice_number = data.get("invoice_number", ""),
            date           = data.get("date", ""),
            total_amount   = data.get("total_amount", ""),
            subtotal       = data.get("subtotal", ""),
            tax_amount     = data.get("tax_amount", ""),
            raw_text       = data.get("raw_text", ""),
            items          = json.dumps(data.get("items", [])),
            confidence     = data.get("confidence", 0.0),
            created_at     = datetime.utcnow()
        )
        session.add(doc)
        session.commit()
        doc_id = doc.id
        print(f"   ✓ Saved to database with ID: {doc_id}")
        return doc_id
    except Exception as e:
        session.rollback()
        print(f"   ⚠ DB save error: {e}")
        return -1
    finally:
        session.close()

def get_all_records() -> List[dict]:
    session = Session()
    try:
        docs = session.query(Document).order_by(Document.created_at.desc()).all()
        return [
            {
                "ID"           : d.id,
                "File"         : d.filename,
                "Type"         : d.document_type,
                "Vendor"       : d.vendor,
                "Invoice #"    : d.invoice_number,
                "Date"         : d.date,
                "Total"        : d.total_amount,
                "Saved At"     : str(d.created_at)[:19] if d.created_at else ""
            }
            for d in docs
        ]
    finally:
        session.close()

print("✅ Database ready!")
print("   File    : tax_documents.db")
print("   Table   : documents")
print("   Columns : id, filename, document_type, vendor, invoice_number,")
print("             date, total_amount, raw_text, items, confidence")

---
# Part 4: OCR Engine
EasyOCR — fast, accurate, no external dependencies.

In [ ]:
# ============================================================================
# CELL 4: INITIALIZE OCR ENGINE (EasyOCR)
# ============================================================================

print("🔍 Initializing EasyOCR Engine...\n")
print("   Language support : English")
print("   GPU              : Auto-detect")
print("   First run        : Downloads ~100MB model\n")

# Detect GPU
use_gpu = torch.cuda.is_available()
print(f"   GPU Available    : {use_gpu}")

# Initialize EasyOCR
ocr_reader = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)

def extract_text_from_image(image_path: str) -> str:
    """Extract raw text from image using EasyOCR"""
    try:
        results = ocr_reader.readtext(image_path, detail=0, paragraph=True)
        text = "\n".join(results)
        return text.strip()
    except Exception as e:
        return f"OCR Error: {e}"

def preprocess_image(image_path: str) -> str:
    """Enhance image for better OCR accuracy"""
    try:
        img = cv2.imread(image_path)
        if img is None:
            return image_path

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Denoise
        denoised = cv2.fastNlMeansDenoising(gray, h=10)

        # Enhance contrast
        clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(denoised)

        # Save preprocessed
        out_path = "/tmp/preprocessed.png"
        cv2.imwrite(out_path, enhanced)
        return out_path
    except:
        return image_path

print("\n✅ EasyOCR Engine Ready!")
print("   Mode    : English, Paragraph grouping")
print("   Preprocessing : Grayscale + Denoise + CLAHE enhancement")

---
# Part 5: Document Classifier
DistilBERT zero-shot classifier — no training required, works immediately.

In [ ]:
# ============================================================================
# CELL 5: DOCUMENT CLASSIFIER (DistilBERT Zero-Shot)
# ============================================================================

print("🤖 Loading Document Classifier...\n")
print("   Model  : facebook/bart-large-mnli (Zero-Shot)")
print("   Labels : Invoice, Receipt, Tax Form, Bank Statement, Other\n")

# Zero-shot classifier — works without training
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1   # CPU
)

DOCUMENT_LABELS = [
    "invoice",
    "receipt",
    "tax form",
    "bank statement",
    "purchase order"
]

def classify_document(text: str) -> dict:
    """Classify document type from extracted text"""
    if not text or len(text.strip()) < 10:
        return {"label": "unknown", "confidence": 0.0}

    try:
        result = classifier(
            text[:512],        # Use first 512 chars
            DOCUMENT_LABELS,
            multi_label=False
        )
        return {
            "label"     : result["labels"][0],
            "confidence": round(result["scores"][0], 3)
        }
    except Exception as e:
        print(f"   ⚠ Classifier error: {e}")
        return {"label": "document", "confidence": 0.5}

print("✅ Classifier Ready!")
print("   Type   : Zero-Shot (no training needed)")
print("   Labels :", DOCUMENT_LABELS)

---
# Part 6: Entity Extraction
Regex-based smart extraction for financial entities.

In [ ]:
# ============================================================================
# CELL 6: FINANCIAL ENTITY EXTRACTOR
# ============================================================================

print("🔍 Setting up Entity Extractor...\n")

def extract_entities(text: str) -> dict:
    """Extract financial entities from OCR text using smart regex"""

    entities = {
        "vendor"         : "",
        "invoice_number" : "",
        "date"           : "",
        "total_amount"   : "",
        "subtotal"       : "",
        "tax_amount"     : "",
        "items"          : []
    }

    lines = text.split("\n")

    # ── Vendor (first non-empty line, or after known keywords) ──
    vendor_keywords = ["from:", "vendor:", "supplier:", "company:", "store:", "shop:"]
    for line in lines[:5]:
        line = line.strip()
        if len(line) > 3 and not re.match(r'^[\d\s\.\-\/]+$', line):
            entities["vendor"] = line
            break
    for line in lines:
        for kw in vendor_keywords:
            if kw in line.lower():
                entities["vendor"] = re.sub(rf'(?i){kw}\s*', '', line).strip()

    # ── Invoice Number ──
    inv_patterns = [
        r'(?i)invoice\s*(?:no|number|#|num)[:\s]*([A-Z0-9\-/]+)',
        r'(?i)inv\s*(?:no|#)[:\s]*([A-Z0-9\-/]+)',
        r'(?i)bill\s*(?:no|#)[:\s]*([A-Z0-9\-/]+)',
        r'(?i)order\s*(?:no|#)[:\s]*([A-Z0-9\-/]+)',
        r'#([A-Z0-9]{4,15})\b',
    ]
    for pattern in inv_patterns:
        match = re.search(pattern, text)
        if match:
            entities["invoice_number"] = match.group(1).strip()
            break

    # ── Date ──
    date_patterns = [
        r'\b(\d{1,2}[\-\/\.\s]\d{1,2}[\-\/\.\s]\d{2,4})\b',
        r'\b(\d{4}[\-\/\.]\d{1,2}[\-\/\.]\d{1,2})\b',
        r'(?i)(\d{1,2}\s+(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{2,4})',
        r'(?i)((?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2},?\s+\d{2,4})',
    ]
    for pattern in date_patterns:
        match = re.search(pattern, text)
        if match:
            entities["date"] = match.group(1).strip()
            break

    # ── Total Amount ──
    total_patterns = [
        r'(?i)total\s*(?:amount)?\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
        r'(?i)grand\s*total\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
        r'(?i)amount\s*due\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
        r'(?i)net\s*(?:payable|total)\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
        r'(?i)balance\s*due\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
    ]
    for pattern in total_patterns:
        match = re.search(pattern, text)
        if match:
            entities["total_amount"] = match.group(1).strip().replace(",", "")
            break

    # ── Subtotal ──
    sub_patterns = [
        r'(?i)sub\s*total\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
        r'(?i)subtotal\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
    ]
    for pattern in sub_patterns:
        match = re.search(pattern, text)
        if match:
            entities["subtotal"] = match.group(1).strip().replace(",", "")
            break

    # ── Tax Amount ──
    tax_patterns = [
        r'(?i)(?:vat|gst|tax|hst|pst)\s*[:\(\d%]*\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
        r'(?i)sales\s*tax\s*[:\$£€Rs\.]*\s*([\d,\.]+)',
    ]
    for pattern in tax_patterns:
        match = re.search(pattern, text)
        if match:
            entities["tax_amount"] = match.group(1).strip().replace(",", "")
            break

    # ── Line Items ──
    item_pattern = r'([A-Za-z][A-Za-z\s]{3,40})\s+[\$£€Rs\.]*\s*(\d+[\.,]\d{2})'
    items = re.findall(item_pattern, text)
    entities["items"] = [f"{item[0].strip()} — {item[1]}" for item in items[:8]]

    return entities


print("✅ Entity Extractor Ready!")
print("   Extracts : vendor, invoice_number, date, total_amount,")
print("              subtotal, tax_amount, line_items")

---
# Part 7: Sample Invoice Generator
Create a test invoice image to demo the system without needing a real document.

In [ ]:
# ============================================================================
# CELL 7: CREATE SAMPLE INVOICE FOR TESTING
# ============================================================================

print("🖼️  Creating Sample Invoice for Testing...\n")

def create_sample_invoice():
    """Generate a realistic sample invoice image"""
    img  = Image.new('RGB', (700, 950), color='white')
    draw = ImageDraw.Draw(img)

    # Try system fonts
    try:
        font_bold  = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
        font_title = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 28)
        font_reg   = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 17)
        font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
    except:
        font_bold  = ImageFont.load_default()
        font_title = font_bold
        font_reg   = font_bold
        font_small = font_bold

    # Header background
    draw.rectangle([(0, 0), (700, 100)], fill='#1a1a2e')
    draw.text((30, 20),  "TECHMART SOLUTIONS PVT LTD",   fill='white', font=font_title)
    draw.text((30, 60),  "www.techmart.com | info@techmart.com", fill='#aaaaaa', font=font_small)
    draw.text((530, 20), "INVOICE",     fill='#f0a500', font=font_title)

    y = 120
    # Invoice meta
    draw.text((30,  y), "Invoice No : INV-2024-00892",      fill='#333', font=font_bold)
    draw.text((400, y), "Date : 15 March 2024",             fill='#333', font=font_bold)
    y += 35
    draw.text((30,  y), "Bill To  : Muhammad Attiq",        fill='#333', font=font_reg)
    draw.text((400, y), "Due Date : 30 March 2024",         fill='#333', font=font_reg)
    y += 25
    draw.text((30,  y), "Address  : Islamabad, Pakistan",   fill='#555', font=font_small)

    # Divider
    y += 40
    draw.rectangle([(30, y), (670, y+2)], fill='#1a1a2e')
    y += 15

    # Table header
    draw.rectangle([(30, y), (670, y+35)], fill='#f0a500')
    draw.text((40,  y+8), "Description",          fill='white', font=font_bold)
    draw.text((370, y+8), "Qty",                  fill='white', font=font_bold)
    draw.text((450, y+8), "Unit Price",            fill='white', font=font_bold)
    draw.text((570, y+8), "Amount",                fill='white', font=font_bold)
    y += 45

    # Line items
    items = [
        ("Laptop - Dell Inspiron 15",    1, 85000),
        ("Wireless Mouse Logitech M185", 2,  2500),
        ("USB-C Hub (7-port)",           1,  4500),
        ("HDMI Cable 2m",                3,   800),
    ]
    for i, (desc, qty, price) in enumerate(items):
        bg = '#f9f9f9' if i % 2 == 0 else 'white'
        draw.rectangle([(30, y), (670, y+32)], fill=bg)
        draw.text((40,  y+7), desc,               fill='#222', font=font_reg)
        draw.text((385, y+7), str(qty),            fill='#222', font=font_reg)
        draw.text((450, y+7), f"Rs. {price:,}",   fill='#222', font=font_reg)
        draw.text((555, y+7), f"Rs. {qty*price:,}", fill='#222', font=font_reg)
        y += 32

    # Totals
    y += 20
    draw.rectangle([(30, y), (670, y+2)], fill='#cccccc')
    y += 15
    subtotal = sum(q*p for _,q,p in items)
    tax      = int(subtotal * 0.17)
    total    = subtotal + tax

    draw.text((450, y),    "Subtotal :",    fill='#333', font=font_bold)
    draw.text((570, y),    f"Rs. {subtotal:,}", fill='#333', font=font_reg)
    y += 30
    draw.text((450, y),    "GST (17%) :",   fill='#333', font=font_bold)
    draw.text((570, y),    f"Rs. {tax:,}",  fill='#333', font=font_reg)
    y += 30
    draw.rectangle([(440, y-5), (670, y+38)], fill='#1a1a2e')
    draw.text((450, y+8),  "TOTAL DUE :",  fill='white', font=font_bold)
    draw.text((555, y+8),  f"Rs. {total:,}", fill='#f0a500', font=font_bold)

    # Footer
    y += 80
    draw.text((30, y),  "Payment Terms: Net 15 Days",  fill='#888', font=font_small)
    draw.text((30, y+20), "Bank: HBL | Account: 1234-5678-9012", fill='#888', font=font_small)
    draw.text((30, y+40), "Thank you for your business!", fill='#1a1a2e', font=font_bold)

    path = "/tmp/sample_invoice.png"
    img.save(path)
    return path

sample_path = create_sample_invoice()
print(f"✅ Sample invoice created: {sample_path}")
print("   You can use this to test the system!")

# Display it
display(Image.open(sample_path).resize((500, 680)))

---
# Part 8: Full Processing Pipeline
Combines OCR → Classification → Entity Extraction → Database Storage.

In [ ]:
# ============================================================================
# CELL 8: FULL DOCUMENT PROCESSING PIPELINE
# ============================================================================

print("⚙️  Building Full Processing Pipeline...\n")

def process_document(image_input, filename: str = "uploaded_image") -> dict:
    """
    Full pipeline:
    Image → Preprocess → OCR → Classify → Extract → Save DB
    """
    result = {
        "filename"       : filename,
        "document_type"  : "unknown",
        "confidence"     : 0.0,
        "vendor"         : "",
        "invoice_number" : "",
        "date"           : "",
        "total_amount"   : "",
        "subtotal"       : "",
        "tax_amount"     : "",
        "items"          : [],
        "raw_text"       : "",
        "db_id"          : -1,
        "processing_time": 0.0,
        "error"          : None
    }

    start = time.time()

    try:
        # ── Step 1: Save to temp ──────────────────────────────────
        temp_path = "/tmp/processing_input.png"
        if isinstance(image_input, np.ndarray):
            Image.fromarray(image_input).save(temp_path)
        elif isinstance(image_input, str):
            temp_path = image_input
        else:
            image_input.save(temp_path)

        print("📸 Step 1/4 : Preprocessing image...")
        preprocessed = preprocess_image(temp_path)

        # ── Step 2: OCR ───────────────────────────────────────────
        print("🔍 Step 2/4 : Extracting text (EasyOCR)...")
        raw_text = extract_text_from_image(preprocessed)
        result["raw_text"] = raw_text
        word_count = len(raw_text.split())
        print(f"            Extracted {word_count} words")

        if not raw_text or word_count < 3:
            result["error"] = "Could not extract text. Check image quality."
            return result

        # ── Step 3: Classify ──────────────────────────────────────
        print("🤖 Step 3/4 : Classifying document type...")
        cls = classify_document(raw_text)
        result["document_type"] = cls["label"]
        result["confidence"]    = cls["confidence"]
        print(f"            Type: {cls['label']} (confidence: {cls['confidence']:.0%})")

        # ── Step 4: Extract entities ──────────────────────────────
        print("💡 Step 4/4 : Extracting financial entities...")
        entities = extract_entities(raw_text)
        result.update(entities)

        # ── Save to DB ────────────────────────────────────────────
        result["processing_time"] = round(time.time() - start, 2)
        result["db_id"] = save_to_db(result)

        print(f"\n✅ Done in {result['processing_time']}s  |  DB ID: {result['db_id']}")

    except Exception as e:
        import traceback
        result["error"] = str(e)
        result["processing_time"] = round(time.time() - start, 2)
        print(f"\n❌ Pipeline error: {e}")
        traceback.print_exc()

    return result


def format_result(result: dict) -> str:
    """Format extraction result for display"""
    if result.get("error"):
        return f"❌ Error: {result['error']}"

    items_text = ""
    if result.get("items"):
        items_text = "\n" + "\n".join(f"     • {item}" for item in result["items"])

    return f"""✅ DOCUMENT PROCESSED SUCCESSFULLY
{'='*55}

📄 Document Info:
   Type          : {result['document_type'].upper()}
   Confidence    : {result['confidence']:.0%}
   Database ID   : #{result['db_id']}
   Processing    : {result['processing_time']}s

🏢 Extracted Data:
   Vendor        : {result['vendor'] or '—'}
   Invoice #     : {result['invoice_number'] or '—'}
   Date          : {result['date'] or '—'}

💰 Financial Data:
   Subtotal      : {result['subtotal'] or '—'}
   Tax           : {result['tax_amount'] or '—'}
   Total Amount  : {result['total_amount'] or '—'}

🛒 Line Items:{items_text if items_text else ' —'}

📝 Raw Text Preview:
{result['raw_text'][:300]}{'...' if len(result['raw_text']) > 300 else ''}
{'='*55}"""


print("✅ Pipeline Ready!")
print("   Steps : Preprocess → OCR → Classify → Extract → Save DB")

---
# Part 9: Quick Test
Test the pipeline with the sample invoice we generated.

In [ ]:
# ============================================================================
# CELL 9: QUICK TEST WITH SAMPLE INVOICE
# ============================================================================

print("🧪 Running Quick Test with Sample Invoice...")
print("="*60 + "\n")

# Process the sample invoice
result = process_document(sample_path, filename="sample_invoice.png")

# Display formatted result
print("\n" + format_result(result))

---
# Part 10: Gradio Interface
Professional web interface with tabs for Upload, Results, and Database.

In [ ]:
# ============================================================================
# CELL 10: GRADIO INTERFACE
# ============================================================================

print("🎨 Building Gradio Interface...\n")

# Custom CSS
custom_css = """
.header-box { background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 20px; border-radius: 10px; }
.result-box { font-family: monospace; font-size: 13px; }
.gr-button-primary { background: #f0a500 !important; border: none !important; }
"""

last_result = {}

def gradio_process(image):
    """Process uploaded image and return formatted results"""
    global last_result
    if image is None:
        return "⚠️ Please upload an image first.", "", ""

    print("\n" + "="*50)
    print("📥 New document received")

    result = process_document(image, filename="user_upload.png")
    last_result = result

    formatted = format_result(result)

    # Summary card
    summary = f"""
📊 Quick Summary
────────────────
Type     : {result['document_type'].upper()}
Vendor   : {result['vendor'] or 'Not detected'}
Total    : {result['total_amount'] or 'Not detected'}
Date     : {result['date'] or 'Not detected'}
DB ID    : #{result['db_id']}
Time     : {result['processing_time']}s
"""

    # Raw text
    raw = result.get("raw_text", "No text extracted")

    return formatted, summary, raw


def gradio_load_sample():
    """Load the sample invoice for demo"""
    return Image.open(sample_path)


def gradio_get_records():
    """Fetch all records from database as formatted text"""
    records = get_all_records()
    if not records:
        return "📭 No records yet. Process a document to get started!"

    lines = [f"📊 DATABASE RECORDS ({len(records)} total)", "="*70]
    for r in records:
        lines.append(
            f"[#{r['ID']}] {r['Type'].upper():15s} | "
            f"Vendor: {r['Vendor'][:20]:20s} | "
            f"Total: {r['Total']:12s} | "
            f"Date: {r['Date']:12s} | "
            f"{r['Saved At']}"
        )
    lines.append("="*70)
    return "\n".join(lines)


# ── BUILD UI ─────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Tax Document Intelligence",
    css=custom_css
) as demo:

    gr.Markdown("""
    # 🧾 Tax Document Intelligence System
    ### AI-Powered Receipt & Invoice Data Extraction
    *Upload any invoice or receipt → Auto-extract vendor, total, date, line items → Save to database*
    ---
    """)

    with gr.Tabs():

        # ── TAB 1: UPLOAD & PROCESS ───────────────────────────────
        with gr.Tab("📤 Upload & Extract"):
            gr.Markdown("### Upload your invoice or receipt image")

            with gr.Row():
                with gr.Column(scale=1):
                    image_input = gr.Image(
                        label="Upload Invoice / Receipt",
                        type="numpy",
                        height=400
                    )
                    with gr.Row():
                        process_btn = gr.Button(
                            "🚀 Extract Data",
                            variant="primary",
                            size="lg"
                        )
                        sample_btn = gr.Button(
                            "📄 Load Sample",
                            variant="secondary",
                            size="lg"
                        )

                    gr.Markdown("""
                    **💡 Tips for best results:**
                    - Use clear, well-lit photos
                    - Ensure all text is readable
                    - Works with JPG, PNG, WEBP
                    - Supports English text
                    """)

                with gr.Column(scale=1):
                    summary_out = gr.Textbox(
                        label="📊 Quick Summary",
                        lines=10,
                        interactive=False,
                        elem_classes="result-box"
                    )
                    raw_text_out = gr.Textbox(
                        label="📝 Extracted Raw Text",
                        lines=10,
                        interactive=False,
                        elem_classes="result-box"
                    )

            full_result_out = gr.Textbox(
                label="📋 Full Extraction Report",
                lines=20,
                interactive=False,
                elem_classes="result-box"
            )

            process_btn.click(
                fn=gradio_process,
                inputs=[image_input],
                outputs=[full_result_out, summary_out, raw_text_out]
            )

            sample_btn.click(
                fn=gradio_load_sample,
                inputs=[],
                outputs=[image_input]
            )

        # ── TAB 2: DATABASE RECORDS ───────────────────────────────
        with gr.Tab("🗄️ Database Records"):
            gr.Markdown("### All processed documents stored in SQLite")

            refresh_btn = gr.Button("🔄 Refresh Records", variant="primary")

            records_out = gr.Textbox(
                label="Database Records",
                lines=20,
                interactive=False,
                elem_classes="result-box",
                value="Click 'Refresh Records' to load data."
            )

            refresh_btn.click(
                fn=gradio_get_records,
                inputs=[],
                outputs=[records_out]
            )

        # ── TAB 3: HOW IT WORKS ───────────────────────────────────
        with gr.Tab("ℹ️ How It Works"):
            gr.Markdown("""
            ## 🔧 System Architecture

            ```
            Image Upload
                │
                ▼
            Image Preprocessing  ← Grayscale + Denoise + CLAHE
                │
                ▼
            EasyOCR Engine       ← Extract raw text
                │
                ▼
            DistilBERT Classifier ← Invoice / Receipt / Tax Form
                │
                ▼
            Regex Entity Extractor ← Vendor, Total, Date, Items
                │
                ▼
            SQLite Database       ← Persistent storage
                │
                ▼
            Gradio UI Display     ← Show results
            ```

            ## 🏗️ Tech Stack

            | Component       | Technology              |
            |-----------------|-------------------------|
            | OCR Engine      | EasyOCR (English)       |
            | Classifier      | DistilBERT Zero-Shot    |
            | Entity Extract  | Smart Regex Patterns    |
            | Database        | SQLite + SQLAlchemy     |
            | Interface       | Gradio                  |
            | Image Processing| OpenCV + PIL            |

            ## 📊 What Gets Extracted

            - **Vendor / Company Name**
            - **Invoice / Receipt Number**
            - **Date**
            - **Subtotal, Tax, Total Amount**
            - **Line Items** (product names + prices)
            - **Document Type** (Invoice / Receipt / Tax Form)

            ## 🚀 Deployment

            - **HuggingFace Space:** https://huggingface.co/spaces/Attiqfr/tax-document-intelligence
            - **Local API:** http://localhost:8000/docs
            - **Database:** SQLite (tax_documents.db)

            ---
            *Built for NLP & MLOps Final Year Project — Bootcamp Cohort 14*
            """)

print("✅ Gradio Interface Built!")

---
# Part 11: Launch Application

In [ ]:
# ============================================================================
# CELL 11: LAUNCH APPLICATION
# ============================================================================

print("\n" + "="*70)
print("🚀 LAUNCHING TAX DOCUMENT INTELLIGENCE SYSTEM")
print("="*70 + "\n")

print("📱 Interface will open below (or in new tab)")
print("🌐 Public share URL will be generated\n")
print("📤 Go to 'Upload & Extract' tab")
print("📄 Click 'Load Sample' to test instantly\n")

demo.launch(
    share      = True,    # public URL
    debug      = True,
    show_error = True,
    quiet      = False
)

---
# 🎉 Congratulations!

You've successfully built and demonstrated the **Tax Document Intelligence System**!

## ✅ What Was Accomplished

### Computer Vision & OCR:
1. ✅ EasyOCR — fast, accurate text extraction
2. ✅ Image preprocessing — grayscale, denoising, CLAHE enhancement
3. ✅ Works on any invoice, receipt, or tax document

### NLP & AI:
4. ✅ DistilBERT Zero-Shot Classifier — no training needed
5. ✅ Smart Regex Entity Extraction — vendor, date, totals, line items
6. ✅ Full end-to-end NLP pipeline

### Database:
7. ✅ SQLite + SQLAlchemy — every extraction saved automatically
8. ✅ Records viewer — browse all processed documents

### Interface:
9. ✅ Professional Gradio UI — 3 tabs
10. ✅ Public URL for sharing

---

## 🚀 Project Links

- **HuggingFace Space:** https://huggingface.co/spaces/Attiqfr/tax-document-intelligence
- **GitHub Repo:** https://github.com/CoderAttiq/tax-document-intelligence
- **Local API Docs:** http://localhost:8000/docs

---
**Built with ❤️ for NLP & MLOps Final Year Project — Bootcamp Cohort 14** 🎓